# Interface Plates — Structural Theory

Validates each plate variant against the two load cases that govern its duty cycle:

1. **Bolt pattern under fault current** — joint heating during 5-cycle bolted-fault clear; joint must stay below 6061-T6 yield-derating threshold (150°C).
2. **Thermal expansion differential** — plate (6061 alpha=23.6e-6/K) vs receiver frame (A36 alpha=11.7e-6/K); per-corner-bolt radial offset must stay within bolt clearance.

Per-plate inputs in `sim/{plate_id}/constants.py`. Solver in `sim/{plate_id}/model.py`. Assertions in `sim/{plate_id}/test_run.py`.

Other load cases (deflection, stress concentration at penetrations, wind load) are bounded by inspection — see ADR-XXX.

## Run all plates

In [1]:
import importlib

PLATES = ["cg", "bg_ac", "ex_g", "ex_c"]
results = {}
for plate in PLATES:
    constants = importlib.import_module(f"sim.{plate}.constants")
    model = importlib.import_module(f"sim.{plate}.model")
    res = model.solve()
    results[plate] = (constants, res)
    print(f"=== {plate.upper()} ===")
    print(f"  joint temp rise  : {res.joint_temp_rise.to(constants.ureg.kelvin):.2f}")
    print(f"  thermal offset   : {res.thermal_offset.to(constants.ureg.mm):.3f}")
    print()


=== CG ===
  joint temp rise  : 50.80 kelvin
  thermal offset   : 0.449 millimeter



=== BG_AC ===
  joint temp rise  : 2.78 kelvin
  thermal offset   : 0.449 millimeter



=== EX_G ===
  joint temp rise  : 50.80 kelvin
  thermal offset   : 0.449 millimeter



=== EX_C ===
  joint temp rise  : 2.78 kelvin
  thermal offset   : 0.449 millimeter



## Verdicts table

In [2]:
print(f"{'plate':<6} {'fault rise (K)':<18} {'fault verdict':<14} {'thermal off (mm)':<18} {'thermal margin (mm)':<22} {'thermal verdict':<14}")
print("-" * 100)
for plate_name, (consts, res) in results.items():
    rise_k = res.joint_temp_rise.to(consts.ureg.kelvin).magnitude
    fault_verdict = "PASS" if rise_k < (consts.JOINT_TEMP_THRESHOLD_C - consts.T_AMBIENT_FAULT_C) else "FAIL"

    offset_mm = res.thermal_offset.to(consts.ureg.mm).magnitude
    clearance_mm = consts.BOLT_CLEARANCE_RADIAL.to(consts.ureg.mm).magnitude
    margin_mm = clearance_mm - offset_mm
    thermal_verdict = "PASS" if margin_mm > 0 else "FAIL"

    print(f"{plate_name:<6} {rise_k:<18.2f} {fault_verdict:<14} {offset_mm:<18.3f} {margin_mm:<22.3f} {thermal_verdict:<14}")


plate  fault rise (K)     fault verdict  thermal off (mm)   thermal margin (mm)    thermal verdict
----------------------------------------------------------------------------------------------------
cg     50.80              PASS           0.449              0.051                  PASS          
bg_ac  2.78               PASS           0.449              0.051                  PASS          
ex_g   50.80              PASS           0.449              0.051                  PASS          
ex_c   2.78               PASS           0.449              0.051                  PASS          


## Design risk surfaced

Thermal margin for both plates is **razor-thin** (~0.05 mm). Real fab tolerance per ISO 2768-m on bolt-hole positions is +/-0.1 mm, which would consume the margin entirely. Two mitigations:

1. **Slot the corner bolt holes** — allow radial expansion. Easy fab change.
2. **Specify tighter tolerance** (ISO 2768-f, +/-0.05 mm) on hole positions. Costs more per fab unit.

Captured for procurement — see ADR-XXX (TBD: write defense variant ADR alongside step 6.9 work).

## Sanity checks

In [3]:
# Sanity 1: fault energy <<< BESS capacity. Should be ~1e-7 ratio.
for plate_name, (consts, _res) in results.items():
    n_eff = consts.BOLT_COUNT / 2
    i_worst = (consts.FAULT_CURRENT.magnitude.nominal_value + consts.FAULT_CURRENT.magnitude.std_dev) * consts.ureg.kA
    r_worst = (consts.R_JOINT_PER_BOLT.magnitude.nominal_value + 2 * consts.R_JOINT_PER_BOLT.magnitude.std_dev) * consts.ureg.microohm
    energy = (i_worst**2 * r_worst / n_eff * consts.FAULT_DURATION).to(consts.ureg.J)
    ratio = (energy / (1.9 * consts.ureg.MWh)).to(consts.ureg.dimensionless)
    print(f"{plate_name.upper()} fault energy: {energy:.1f} ; ratio to 1.9 MWh BESS: {ratio:.2e}")


CG fault energy: 5434.9 joule ; ratio to 1.9 MWh BESS: 7.95e-07 dimensionless
BG_AC fault energy: 297.0 joule ; ratio to 1.9 MWh BESS: 4.34e-08 dimensionless
EX_G fault energy: 5434.9 joule ; ratio to 1.9 MWh BESS: 7.95e-07 dimensionless
EX_C fault energy: 297.0 joule ; ratio to 1.9 MWh BESS: 4.34e-08 dimensionless


In [4]:
# Sanity 2: differential expansion order of magnitude.
for plate_name, (consts, _res) in results.items():
    delta_alpha = consts.PLATE_ALPHA - consts.FRAME_ALPHA
    per_meter = (delta_alpha * 1000 * consts.ureg.mm * 85 * consts.ureg.kelvin).to(consts.ureg.mm)
    print(f"{plate_name.upper()} differential expansion per meter at deltaT=85K: {per_meter:.3f}")


CG differential expansion per meter at deltaT=85K: 1.012 millimeter
BG_AC differential expansion per meter at deltaT=85K: 1.012 millimeter
EX_G differential expansion per meter at deltaT=85K: 1.012 millimeter
EX_C differential expansion per meter at deltaT=85K: 1.012 millimeter
